# Model Selection and Evaluation Workflow
### Internship Track: Week 2 – Task 4
**Author:** Sayantani Adhikary  
**Objective:** Benchmark and evaluate Logistic Regression, Decision Tree, and Random Forest models using 5-Fold Cross-Validation, analyze variance/bias trade-offs, and select the optimal classifier for deployment.


## 1. Import Libraries & Load Data
We import Pandas, NumPy, Scikit-Learn model selection tools, classifiers, and Seaborn for visualization.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
df = pd.read_csv('../data/dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


## 2. Data Preprocessing & Train-Test Split
Separate features ($X$) and target ($y$), apply `StandardScaler`, and reserve 20% of data for final holdout testing.


In [1]:
X = df.drop(columns=['Approved'])
y = df['Approved']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training instances: {X_train.shape[0]}, Test instances: {X_test.shape[0]}")


## 3. 5-Fold Stratified Cross-Validation
We evaluate candidate model generalization across 5 cross-validation folds on the training set to prevent overfitting.


In [1]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}: CV Mean = {scores.mean()*100:.2f}%, Std = {scores.std():.4f}")


## 4. Cross-Validation Performance Visualization
Plot cross-validation score distributions across folds using box plots to evaluate model stability.


In [1]:
cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=cv_df, palette='Set2')
plt.title('5-Fold Cross-Validation Accuracy Spread Across Models', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy Score')
plt.xlabel('Algorithm Architecture')
plt.tight_layout()
plt.show()


## 5. Holdout Test Set Evaluation & Comparison
Fit each model on the full training set and evaluate performance on the unseen test set.


In [1]:
test_scores = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    test_scores.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, preds, average='weighted'),
        'F1-Score': f1_score(y_test, preds, average='weighted')
    })

comparison_table = pd.DataFrame(test_scores)
print("=== Holdout Test Performance Comparison ===")
print(comparison_table.to_string(index=False))


## 6. Selected Model Confusion Matrix
Examine the confusion matrix of the selected champion model (**Random Forest**).


In [1]:
rf_model = models['Random Forest']
rf_preds = rf_model.predict(X_test)
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
plt.title('Random Forest Test Set Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf_preds, target_names=['Rejected', 'Approved']))
